In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import SVD
from pathlib import Path

In [ ]:
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Project directories
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
netflix_dataset = pd.read_csv(DATA_DIR / "combined_data_1.csv")

In [ ]:
netflix_dataset.head()

In [ ]:
netflix_dataset.tail()

In [ ]:
netflix_dataset.info()

In [ ]:
netflix_dataset.shape

In [ ]:
netflix_dataset.dtypes

In [ ]:
netflix_dataset.isnull().sum()

In [ ]:
# Finding the movie count by assumiung that if there is NaN in Rating column so in front of it , there is a movie id

In [ ]:
# Getting the total number of movie id present in data, which is nan values in rating column will show how many movie are there.. 

movie_count=netflix_dataset.isnull().sum()
movie_count = movie_count["Rating"]
movie_count

In [ ]:
# Finding the total unqiue customer id by substracting the movie count from uniuque values in cust id column

In [ ]:
# To claculate how many customers we are having in the dataset
customer_count=netflix_dataset['Cust_Id'].nunique()

In [ ]:
customer_count - movie_count 

**we have totak 4 lakh , 70 thousand and 7 hundred 58 unique customers and 4499 unique movie id**

In [ ]:
rating_count = netflix_dataset['Cust_Id'].count() - movie_count
rating_count

In [ ]:
rating_group = netflix_dataset.groupby("Rating")['Rating'].count()

In [ ]:
rating_group

**Data Visualization
**

In [ ]:
REPORTS_DIR

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Rating', data=netflix_dataset, palette='viridis')
plt.title("Distribution of Ratings", fontsize=14, fontweight='bold')
plt.xlabel("Rating (stars)")
plt.ylabel("Number of Ratings")

# Annotate each bar with its count — makes the chart self-explanatory
# without needing the reader to squint at the y-axis
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height()):,}',
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=9)

plt.tight_layout()
# plt.savefig(REPORTS_DIR, dpi=150)  # save for README embedding
plt.show()

In [ ]:
netflix_dataset

In [ ]:
# Data Cleaning

In [ ]:
# def get_movie_id_col(data):
#     movie_id = None
#     movie_np = []
#     for cust_id in data['Cust_Id']:
#         if ":" in cust_id:
#             movie_id = int(cust_id.replace(":",""))
#         movie_np.append(movie_id)
#     return movie_np

In [ ]:
# movieId_list = get_movie_id_col(netflix_dataset)   # This is easy to understand but slow to process..

In [ ]:
is_marker = netflix_dataset['Cust_Id'].str.contains(":")

In [ ]:
netflix_dataset['Movie_id'] = np.where(is_marker, netflix_dataset['Cust_Id'], np.nan)     # Where ever the is_marker is True in Movie_id put there the Cust_id value and where ever there is False put there np.nan

In [ ]:
netflix_dataset.head()

In [ ]:
netflix_dataset['Movie_id'] = netflix_dataset['Movie_id'].ffill()  # It will add the value in the Movie_id by propogating the last Movie_id value

In [ ]:
netflix_dataset

In [ ]:
netflix_dataset['Movie_id'] = netflix_dataset['Movie_id'].str.replace(":","")    # Replace the ":" to " "

In [ ]:
netflix_dataset.info()

In [ ]:
netflix_dataset['Movie_id'] = netflix_dataset['Movie_id'].astype('int')

In [ ]:
netflix_dataset['Movie_id'].nunique()

In [ ]:
netflix_dataset = netflix_dataset[netflix_dataset['Rating'].notna()]   # Simple getting the particular data where Rating is present and if there rating is Nan leave it.

In [ ]:
netflix_dataset.rename(columns={'Cust_Id': 'User_Id'}, inplace=True)

In [ ]:
netflix_dataset.reset_index(drop=True, inplace=True)

In [ ]:
netflix_dataset.info()

In [ ]:
netflix_dataset['User_Id'] = netflix_dataset['User_Id'].astype("int")

In [ ]:
netflix_dataset.info()

Pre-Filtering -  In Which we simply Remove the particular User who have rated less movies and Movies that have rated less in numbers.

In [ ]:
# Finding out Which movies has been less rated for filtering-out

In [ ]:
movies_rate_summary = netflix_dataset.groupby('Movie_id')["Rating"].count()

In [ ]:
movies_rate_summary.describe()

In [ ]:
movies_rate_summary

In [ ]:
movies_rate_summary.sort_values(ascending=False)

In [ ]:
movies_rate_summary = pd.DataFrame(movies_rate_summary)

In [ ]:
movies_rate_summary

In [ ]:
plt.boxplot(movies_rate_summary)
plt.xlabel("Movies Id")
plt.ylabel("No.of.Rating")
plt.show()

In [ ]:
# Now we are creating an Benchmarks which tell that which movie should be selected and which movies should be filter-out


movie_benchmark = round(movies_rate_summary['Rating'].quantile(0.6), 0)

In [ ]:
movie_benchmark

In [ ]:
reference_value = movie_benchmark
plt.boxplot(movies_rate_summary)
plt.title("Movies and Total No-of-Rating")
plt.xlabel("Movies Id")
plt.ylabel("No.of.Rating")
plt.axhline(y=reference_value, color='red', linestyle='--', linewidth=1,
            label=f'Reference = {reference_value}')
plt.show()

In [ ]:
# So here the Movies whose comes below the red line are going to filter out..

In [ ]:
movies_rate_summary.head()

In [ ]:
movies_rate_summary.tail()

In [ ]:
non_drop_movie_list  = movies_rate_summary[movies_rate_summary['Rating'] >= movie_benchmark].index

In [ ]:
non_drop_movie_list      # This hold the Movies ids whose rating is lesser then Benchmark (908) or which comes below the 60% quantile

In [ ]:
len(non_drop_movie_list)

In [ ]:
# Now let see about the customer

In [ ]:
#now we will remove all the users that are in-active
dataset_cust_summary=netflix_dataset.groupby('User_Id')['Rating'].agg(["count"])
dataset_cust_summary

In [ ]:
dataset_cust_summary['count'].describe()

In [ ]:
dataset_cust_summary.sort_values(by= 'count' ,ascending=False)

In [ ]:
cust_benchmark=round(dataset_cust_summary['count'].quantile(0.6),0)
cust_benchmark

In [ ]:
reference_value = cust_benchmark
plt.boxplot(dataset_cust_summary)
plt.title("User and Total No-of-Rating")
plt.xlabel("User Id")
plt.ylabel("No.of.Rating")
plt.axhline(y=reference_value, color='red', linestyle='--', linewidth=1,
            label=f'Reference = {reference_value}')
plt.show()

In [ ]:
non_drop_cust_list=dataset_cust_summary[dataset_cust_summary['count'] >= cust_benchmark].index
non_drop_cust_list

In [ ]:
len(non_drop_cust_list)

In [ ]:
len(non_drop_movie_list)

In [ ]:
# Here are the total User and Movie which is going to be filter out...from the dataset

**Filter Out the Movies whose no.of.Rating is lesser then 908.0 and User who rated the movies less than 36**

In [ ]:
netflix_dataset

Dataset Sturcture before Filtering

In [ ]:
netflix_dataset.shape

In [ ]:
netflix_dataset['Movie_id'].nunique()

In [ ]:
netflix_dataset['User_Id'].nunique()

In [ ]:
netflix_dataset['Movie_id'].isna().sum()

Why we are Filtering? 
470,758 users
4,499 movies
24,053,764 ratings

The problem is that many users may have rated only a handful of movies, and some movies may have received very few ratings.

For collaborative filtering, extremely low-activity users/items provide relatively little information. So one common strategy is:

Keep users and movies that have enough ratings to participate meaningfully in the model.

In [ ]:
# The drop_movie_list contain the index of movie id whose rating is greater then 908 which we will keep in the Netflix Dataset
# The drop_cust_list  contain the index of user id whose had given rating to the movie is greater then atleast 36 which we will keep in the Netflix Dataset

In [ ]:
netflix_dataset = netflix_dataset[netflix_dataset['Movie_id'].isin(non_drop_movie_list)]
netflix_dataset = netflix_dataset[netflix_dataset['User_Id'].isin(non_drop_cust_list)]

In [ ]:
netflix_dataset.shape

In [ ]:
netflix_dataset.reset_index(inplace=True)

In [ ]:
netflix_dataset.drop(columns='index', axis=1, inplace=True)

In [ ]:
netflix_dataset['Movie_id'].nunique()

In [ ]:
netflix_dataset['User_Id'].nunique()

Before/After Filtering Comparison (shows your 81.88% retention story)

In [ ]:
categories = ['Movies', 'Users']
before = [4499, 470758]
after = [1800, 188716]

x = range(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
bars1 = ax.bar([i - width/2 for i in x], before, width, label='Before Filtering', color='#4C72B0')
bars2 = ax.bar([i + width/2 for i in x], after, width, label='After Filtering', color='#DD8452')

ax.set_ylabel('Count')
ax.set_title('Catalog Size Before vs. After Popularity/Activity Filtering', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

# Label each bar with its exact value + retention %
for bars, values in [(bars1, before), (bars2, after)]:
    for bar, val in zip(bars, values):
        ax.annotate(f'{val:,}', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=9)

plt.figtext(0.5, -0.02, "81.88% of all ratings retained despite ~60% reduction in catalog/user count",
            ha='center', fontsize=10, style='italic')
plt.tight_layout()
plt.show()

In [ ]:
retention_pct = (netflix_dataset.shape[0] / 24053764) * 100
print(f"Retained: {retention_pct:.2f}% of original ratings")

# MODEL Building

In [ ]:
movie_title_dataset = pd.read_csv(
    DATA_DIR / "movie_titles.csv",
    encoding="latin",
    header=None,
    usecols=[0, 1, 2],
    names=["Movie_Id", "Year", "Name"]
)

In [ ]:
movie_title_dataset

In [ ]:
movie_title_dataset.shape

In [ ]:
movie_title_dataset.info()

In [ ]:
movie_title_dataset.isna().sum()

In [ ]:
# So, when surprise library when to have the know about the data it was to work with...So here the reader function help to read the data in particular format and the format help to understand the surprire the data

In [ ]:
# Reader help to understand the format of the data or structure of the data. so it help surprise library to work with that particular format of data..
# Dataset this function help us to load our data set into the surprise library.
# SVD is Singular Vector Decomposition 
# cross_validate 

In [ ]:
from surprise import Reader, Dataset, SVD    # Reader 
from surprise.model_selection import cross_validate
from surprise import BaselineOnly

In [ ]:
reader = Reader()

In [ ]:
netflix_dataset

In [ ]:
sample_df = netflix_dataset.sample(n=1_000_000, random_state=42)
data = Dataset.load_from_df(sample_df[['User_Id', 'Movie_id', 'Rating']], reader)

In [ ]:
Svd_model = BaselineOnly()

In [ ]:
cross_validate(Svd_model,data, measures=['RMSE'], cv =3)

In [ ]:
Svd_model = SVD()

In [ ]:
cross_validate(Svd_model,data, measures=['RMSE'], cv =3)

Hyper-Parameter Tuning

In [ ]:
from surprise.model_selection import GridSearchCV

In [ ]:
# surprise default parameter is set by (n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02) are generic,

In [ ]:
paramter_grid = {
    'n_factors': [20,50, 100],
    'n_epochs':[20,30],
    'lr_all' : [0.005, 0.01 ],
    "reg_all" : [0.02, 0.1]
}

In [ ]:
gridsearch = GridSearchCV(SVD, paramter_grid, measures=['rmse'], cv = 3)

In [ ]:
gridsearch.fit(data)

In [ ]:
print(gridsearch.best_score['rmse'])
print(gridsearch.best_params['rmse'])

In [ ]:
# Here what we can  interprete is :- 

# n_factors': 20 (lowest option in your grid, not 50 or 100) — fewer latent dimensions won. 
# This usually signals the data doesn't support very high-dimensional taste patterns at this sample size/density; more factors would likely just overfit noise. 
# Worth trying n_factors: [5, 10, 20] in a follow-up grid to see if even fewer wins — that's a legitimate thing to explore and mention as a "future work" note.

In [ ]:
# reg_all': 0.1 (the higher regularization option) winning over 0.02 reinforces the same story: the model wants more regularization, i.e., it's prone to overfitting on this sample size, so stronger penalties help generalization.

In [ ]:
# n_epochs': 30 (the higher option) winning suggests the model was still improving at 20 epochs — worth a note that a wider grid ([30, 40, 50]) might squeeze out slightly more, though with diminishing returns.

SVD achieves great accuracy!

In [ ]:
print(gridsearch.best_params['rmse'])

In [ ]:
Final_Svd_Model = SVD(n_factors=20,n_epochs=30, lr_all=0.005 , reg_all= 0.1)

In [ ]:
print(gridsearch.best_score['rmse'])

In [ ]:
cross_validate(Final_Svd_Model,data, measures=['RMSE'], cv =3)

In [ ]:
import pickle

model_path = MODEL_DIR / "Final_Svd_Model.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(Final_Svd_Model, f)

In [ ]:
import pickle

with open(model_path, 'rb') as f:
    Final_Svd_Model = pickle.load(f)

Use Final-model for Making Recommendation for Specific User

In [ ]:
netflix_dataset

In [ ]:
movie_title_dataset

In [ ]:
Random_user = netflix_dataset['User_Id'].sample(n = 1, random_state = 42)
Random_user.values

In [ ]:
netflix_dataset.iloc[8210437 : 8210437 + 1,:]    # This is the particular user we are going to recommmand movie too

In [ ]:
user_ratings = netflix_dataset[netflix_dataset['User_Id'] == int(Random_user.values)] 

In [ ]:
user_ratings   # The User rating to all the movies

In [ ]:
# Find out the number of unique movies rated by the user
movies_rated_by_user = user_ratings['Movie_id'].nunique() 
print(f"USer {(Random_user.values)} has rated {movies_rated_by_user} Unique Movies.")

In [ ]:
# So, based on this user taste in the movie we are going to recommand this user some movies for here future reference.

In [ ]:
copy_of_title_data = movie_title_dataset.copy()

In [ ]:
copy_of_title_data.head()

In [ ]:
Clean_copy_of_title_data = copy_of_title_data[copy_of_title_data['Movie_Id'].isin(non_drop_movie_list)]

In [ ]:
Clean_copy_of_title_data

In [ ]:
netflix_dataset

In [ ]:
# Here What we have did is simple which is, in Clean_copy_of_title_data there will be only does movies information (Year, Name) which is not going to be drop...
# Why droping? - as we had did droping the movies id which has less rating so we have to do this in movie-title dataset too...like including only hit movie in the movie title dataset.

In [ ]:
user = int(Random_user.values)

In [ ]:
Clean_copy_of_title_data['User_Preferences_score'] = Clean_copy_of_title_data['Movie_Id'].apply(lambda x : Final_Svd_Model.predict(user, x).est)

In [ ]:
Clean_copy_of_title_data

In [ ]:
# Here What we are doing is We are simple user the SVD(recommandation model) to recommand movie for the user
# How it is estimating - for the particular User model giving Estimaited Value that tell the how much will the movies the user will enjoy...
# The more User_Preferences_score (estimated_score) of the movie for the user the  movie that will should recommand to that User.

In [ ]:
Clean_copy_of_title_data.head()

In [ ]:
Clean_copy_of_title_data.tail()

In [ ]:
Clean_copy_of_title_data.sort_values(by='User_Preferences_score',ascending=False)

In [ ]:
Clean_copy_of_title_data.sort_values(by='User_Preferences_score',ascending=False).head(10)

This Ten movies can be the next ten best rated recommending movie that should be recommanded to this particular user..

In [ ]:
user_ratings

In [ ]:
user_rated_movie = user_ratings['Movie_id']

In [ ]:
Clean_copy_of_title_data

In [ ]:
Clean_copy_of_title_data = Clean_copy_of_title_data[~Clean_copy_of_title_data['Movie_Id'].isin(user_rated_movie)]

In [ ]:
Clean_copy_of_title_data

In [ ]:
# Now We got exaclty movie that we should recommand to the User

In [ ]:
Clean_copy_of_title_data.sort_values(by = 'User_Preferences_score',ascending=False).head()

In [ ]:
# This are the top five movies that user will like to watch

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.recommend import recommend_for_user

In [ ]:
recommend_for_user(831103,Final_Svd_Model,netflix_dataset,movie_title_dataset,non_drop_movie_list,10)

In [ ]:
# This is the Final 5 file that user [831103] should be get recommended at the end

In [ ]:
netflix_dataset = pd.read_csv(DATA_DIR / "combined_data_1.csv")

In [ ]:
netflix_dataset

In [ ]:
from src.data_prep import clean_netflix_data

In [ ]:
clean_netflix_data , non_drop_movie_list, non_drop_cust_list = clean_netflix_data(netflix_dataset)

In [ ]:
clean_netflix_data

In [ ]:
clean_netflix_data.info()

In [ ]:
clean_netflix_data.isna().sum()

Precision@K / Recall@K

In [ ]:
# get a proper train/test split

In [ ]:
data  # This is the data that the Recommednaton is trained on

In [ ]:
from surprise.model_selection import train_test_split

In [ ]:
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

In [ ]:
Final_Svd_Model.fit(trainset)
predictions = Final_Svd_Model.test(testset)

notes :- predictions here is a list of Prediction objects, one per (user, item) pair in your test set — each has .uid (user), .iid (item/movie), .r_ui (true rating), and .est (predicted rating).

In [ ]:
predictions[0]

Here, prediction[0] holds the value of-  
- uid - user
- iid - movie
- r_ui - actual rating
- est - prediction rating 

In [ ]:
# group predictions by user

from collections import defaultdict

def get_top_n_by_user(predictions, n=10):
    user_predictions = defaultdict(list)
    for pred in predictions:
        user_predictions[pred.uid].append((pred.iid, pred.est, pred.r_ui))
    return user_predictions

In [ ]:
def precision_recall_at_k(user_predictions, k=10, threshold=4):
    precisions = {}
    recalls = {}

    for uid, ratings in user_predictions.items():
        if len(ratings) < k:
            continue  # skip users who don't have enough test items to meaningfully evaluate at this k

        ratings.sort(key=lambda x: x[1], reverse=True)
        top_k = ratings[:k]

        relevant_and_recommended = sum((true_r >= threshold) for (_, _, true_r) in top_k)
        total_relevant = sum((true_r >= threshold) for (_, _, true_r) in ratings)

        precisions[uid] = relevant_and_recommended / k
        if total_relevant != 0:
            recalls[uid] = relevant_and_recommended / total_relevant

    return precisions, recalls

In [ ]:
user_predictions = get_top_n_by_user(predictions, n=None)  # gather ALL, not just top-n, for this step
precisions, recalls = precision_recall_at_k(user_predictions, k=10, threshold=4)

avg_precision = sum(precisions.values()) / len(precisions)
avg_recall = sum(recalls.values()) / len(recalls)

print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")

- Precision@10 = : of the top-10 movies your model recommended per user, on average only about 47.5% (roughly 1 out of 10) were movies that user actually rated ≥4 in their test set

- Recall@10 = - of all the movies each user rated ≥4 anywhere in their test set, your top-10 list captured 93% of them. Recall near-perfect at 1.0 essentially means "your top-10 basically contains everything relevant that existed"

In [ ]:
test_sizes = [len(ratings) for ratings in user_predictions.values()]
print("Average test items per user:", sum(test_sizes) / len(test_sizes))
print("Median test items per user:", sorted(test_sizes)[len(test_sizes)//2])

In [ ]:
precisions5, recalls5 = precision_recall_at_k(user_predictions, k=5, threshold=4)
print(len(precisions5), "out of", len(user_predictions), "users evaluated")
print(f"Precision@5: {sum(precisions5.values())/len(precisions5):.4f}")
print(f"Recall@5: {sum(recalls5.values())/len(recalls5):.4f}")

In [ ]:
precisions10, recalls10 = precision_recall_at_k(user_predictions, k=10, threshold=4)
print(len(precisions10), "out of", len(user_predictions), "users evaluated")
print(f"Precision@5: {sum(precisions10.values())/len(precisions10):.4f}")
print(f"Recall@5: {sum(recalls10.values())/len(recalls10):.4f}")

In [ ]:
k_values = ['K=5', 'K=10']
precision_vals = [0.5482, 0.4706]
recall_vals = [0.9127, 0.9336]

x = range(len(k_values))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar([i - width/2 for i in x], precision_vals, width, label='Precision@K', color='#55A868')
ax.bar([i + width/2 for i in x], recall_vals, width, label='Recall@K', color='#C44E52')

ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Precision vs. Recall at Different K', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(k_values)
ax.legend()

for i, (p, r) in enumerate(zip(precision_vals, recall_vals)):
    ax.annotate(f'{p:.2f}', (i - width/2, p), ha='center', va='bottom', fontsize=9)
    ax.annotate(f'{r:.2f}', (i + width/2, r), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()